# Task 3: Correlation Between News Sentiment and Stock Movement

This notebook links headline sentiment to daily stock returns. It normalizes publication dates, aligns news to valid trading days, scores headline sentiment, aggregates multiple articles per stock/day, calculates daily returns, and measures Pearson correlation between sentiment and price movement.

## Tool Selection

VADER is used for sentiment scoring when available because financial headlines are short, polarity-heavy snippets. VADER produces a normalized compound score that is easy to aggregate by ticker/date. A lightweight lexicon fallback is available so the notebook remains reproducible in environments where optional NLP packages are not installed.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_news
from src.technical_indicators import clean_price_data, load_price_data
from src.sentiment_correlation import (
    aggregate_daily_sentiment,
    align_news_to_trading_day,
    average_return_by_sentiment,
    compute_daily_returns,
    merge_sentiment_with_returns,
    pearson_correlation,
    score_headlines,
    sentiment_tool_info,
)

sns.set_theme(style="whitegrid", palette="Set2")
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load News and Price Data

The news dataset should contain `headline`, `date`, and `stock`. The price dataset should contain daily OHLCV data with `date`, `stock`, and preferably `adj_close`.

In [ ]:
news = load_news(PROJECT_ROOT / "data" / "raw")
prices = clean_price_data(load_price_data(PROJECT_ROOT / "data" / "raw"))

print(f"News rows: {len(news):,}")
print(f"Price rows: {len(prices):,}")
display(news.head())
display(prices.head())

## 2. Score Headline Sentiment

Each headline receives a continuous sentiment score and a label: negative, neutral, or positive.

In [ ]:
tool = sentiment_tool_info()
print(tool.name)
print(tool.rationale)

scored_news = score_headlines(news)
scored_news[['date', 'stock', 'headline', 'sentiment_score', 'sentiment_label', 'sentiment_tool']].head()

## 3. Align News Dates to Trading Days

News published on weekends or holidays is aligned to the next available trading day for the same ticker. This prevents non-trading publication dates from being lost during the return join.

In [ ]:
aligned_news = align_news_to_trading_day(scored_news, prices)
daily_sentiment = aggregate_daily_sentiment(aligned_news)

display(aligned_news[['date', 'publication_date', 'trading_date', 'stock', 'sentiment_score']].head())
display(daily_sentiment.head())

## 4. Calculate Daily Returns

Daily returns are calculated as percentage change in adjusted closing price: `(Adj Close_t - Adj Close_t-1) / Adj Close_t-1 * 100`.

In [ ]:
daily_returns = compute_daily_returns(prices)
daily_returns.head()

## 5. Correlate Sentiment and Returns

The merged table contains one row per ticker/trading day where both average sentiment and daily return are available.

In [ ]:
merged = merge_sentiment_with_returns(daily_sentiment, daily_returns)
correlations = pearson_correlation(merged)

display(merged.head())
display(correlations.head(20))

## 6. Visualize the Relationship

The scatter plot shows sentiment-return association. The bar chart compares average returns after negative, neutral, and positive sentiment days.

In [ ]:
overall_corr = merged['avg_sentiment'].corr(merged['daily_return_pct'], method='pearson') if len(merged) >= 2 else float('nan')

fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(data=merged, x='avg_sentiment', y='daily_return_pct', ax=ax, scatter_kws={'alpha': 0.55}, line_kws={'color': '#E45756'})
ax.set_title(f'Sentiment vs Daily Return (Pearson r = {overall_corr:.3f})')
ax.set_xlabel('Average daily sentiment score')
ax.set_ylabel('Daily return (%)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task_3_sentiment_return_scatter.png', dpi=150)
plt.show()

In [ ]:
returns_by_sentiment = average_return_by_sentiment(merged)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=returns_by_sentiment, x='sentiment_label', y='avg_daily_return_pct', ax=ax, palette={'negative': '#E45756', 'neutral': '#BAB0AC', 'positive': '#54A24B'})
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Daily Return by Sentiment Category')
ax.set_xlabel('Sentiment category')
ax.set_ylabel('Average daily return (%)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task_3_return_by_sentiment.png', dpi=150)
plt.show()

returns_by_sentiment

## Interpretation Guide

A positive Pearson coefficient indicates that higher average news sentiment tends to coincide with higher same-day trading returns. A negative coefficient indicates that higher sentiment tends to coincide with lower returns. Values near zero suggest weak linear association in this same-day setup.

Limitations: same-day correlation does not prove causation. Price movement may be driven by earnings, macro news, sector shocks, liquidity, or analyst behavior. Future analysis should test lagged windows, after-hours alignment, topic-specific sentiment, and out-of-sample performance.